# LeetCode 497 — Random Point in Non-overlapping Rectangles bằng Python

Link đề: https://leetcode.com/problems/random-point-in-non-overlapping-rectangles/ · Độ khó: **Medium** · Chủ đề: Randomized, Prefix Sum, Binary Search, Reservoir Sampling, Math

Skeleton LeetCode:

```python
class Solution:

    def __init__(self, rects: List[List[int]]):
        ...

    def pick(self) -> List[int]:
        ...


# Your Solution object will be instantiated and called as such:
# obj = Solution(rects)
# param_1 = obj.pick()
```

Ghi chú: phần phát biểu đề được dịch sang tiếng Việt từ đề gốc. Đây là **bài ngẫu nhiên**: đầu ra của ví dụ trong đề chỉ để minh họa (mỗi lần `pick` ra điểm khác nhau), nên ta **không so đáp án cứng** mà kiểm tra (1) điểm trả về nằm trong đúng một hình chữ nhật và (2) phân bố đều trên các điểm nguyên bằng thống kê chi-bình-phương với seed cố định. Phần "các hình chữ nhật không giao nhau" được hiểu là chúng **không có chung điểm nguyên nào** (kể cả biên) — nên đối chiếu lại đề gốc nếu muốn chắc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chọn **ngẫu nhiên đều** một **điểm nguyên** (tọa độ nguyên) nằm trong **hợp** các hình chữ nhật, **biên được tính**, nên số điểm của một hình chữ nhật `[a, b, x, y]` là `(x - a + 1) * (y - b + 1)`.
2. Nắm công cụ Python: module `random` (`seed`, `randint`, `randrange`, `random`, đối tượng `random.Random`), `bisect`, cộng dồn (prefix sum), `divmod` / `//` / `%` để chuyển một số nguyên thành tọa độ 2 chiều.
3. Đi từ **liệt kê mọi điểm rồi `random.choice`** → **chọn hình chữ nhật theo trọng số bằng quét tuyến tính** → **cộng dồn + tìm kiếm nhị phân `O(log k)`** → **một số ngẫu nhiên duy nhất rồi `divmod`** → **reservoir sampling có trọng số**.
4. Hiểu mấu chốt: để điểm nào cũng đồng khả năng, **xác suất chọn một hình chữ nhật phải tỉ lệ với số điểm nguyên của nó** (không phải diện tích hình học, cũng không phải 1/k); rồi trong hình chữ nhật đó chọn đều từng tọa độ.
5. Biết **kiểm thử chương trình ngẫu nhiên**: seed cố định, kiểm tra tính chất (điểm nằm đúng một hình), kiểm tra phân bố bằng thống kê (chi-bình-phương), và biết một cài đặt sai (đếm thiếu biên, chọn đều hình chữ nhật) bị thống kê phát hiện thế nào.

## 2. Đề bài & ví dụ

Cho mảng các hình chữ nhật **song song với trục tọa độ** và **không chồng lấn**: `rects[i] = [a_i, b_i, x_i, y_i]`, trong đó `(a_i, b_i)` là **góc dưới-trái** và `(x_i, y_i)` là **góc trên-phải** của hình chữ nhật thứ `i`.

Mọi **điểm nguyên** (tọa độ là số nguyên) nằm trong phần mặt phẳng được phủ bởi các hình chữ nhật phải có **xác suất được trả về bằng nhau**.

Hãy cài đặt lớp `Solution`:

- `Solution(int[][] rects)` — khởi tạo với các hình chữ nhật.
- `int[] pick()` — trả về một điểm nguyên `[u, v]` nằm trong phần mặt phẳng được phủ bởi một trong các hình chữ nhật.

**Ví dụ (minh họa — kết quả ngẫu nhiên):**

```
Solution solution = new Solution([[-2, -2, 1, 1], [2, 2, 4, 6]]);
solution.pick(); // [1, -2]
solution.pick(); // [1, -1]
solution.pick(); // [-1, -2]
solution.pick(); // [-2, -2]
solution.pick(); // [0, 0]
```

**Ràng buộc:**

- `1 <= rects.length <= 100`
- `rects[i].length == 4`
- `-10^9 <= a_i < x_i <= 10^9`, `-10^9 <= b_i < y_i <= 10^9`
- `x_i - a_i <= 2000`, `y_i - b_i <= 2000`
- Các hình chữ nhật không chồng lấn.
- Có tối đa `10^4` lần gọi `pick`.

In [1]:
import random
import time
from bisect import bisect_left, bisect_right
from typing import List

rects_demo = [[-2, -2, 1, 1], [2, 2, 4, 6]]
for a, b, x, y in rects_demo:
    w, h = x - a + 1, y - b + 1
    print(f"hình [{a},{b},{x},{y}]: rộng {w} điểm, cao {h} điểm, tổng {w * h} điểm nguyên")

hình [-2,-2,1,1]: rộng 4 điểm, cao 4 điểm, tổng 16 điểm nguyên
hình [2,2,4,6]: rộng 3 điểm, cao 5 điểm, tổng 15 điểm nguyên


### 2.1. Đọc kỹ đề

- **Điểm nguyên bao gồm biên.** Hình `[-2,-2,1,1]` có các hoành độ `-2,-1,0,1` (**4** giá trị, không phải `1 - (-2) = 3`) và cũng 4 tung độ ⇒ `4 × 4 = 16` điểm. Đây là lỗi "hàng rào" (fencepost): đếm số nguyên trong đoạn `[lo, hi]` là `hi - lo + 1`.
- Hình thứ hai `[2,2,4,6]` có `3 × 5 = 15` điểm. Tổng `31` điểm nguyên; mỗi điểm phải có xác suất `1/31`.
- "Đồng khả năng" áp dụng cho **điểm nguyên**, không phải cho hình chữ nhật: hình to phải được chọn nhiều hơn hình nhỏ theo **tỉ lệ số điểm**.
- `x_i - a_i <= 2000` ⇒ mỗi hình có tối đa `2001 × 2001 ≈ 4 triệu` điểm; `100` hình ⇒ tối đa gần `4 × 10^8` điểm. **Không thể** liệt kê hết (hết bộ nhớ).
- Tọa độ có thể lên tới `±10^9`, Python không tràn số nên tính diện tích không lo tràn (Java/C++ cần cẩn thận kiểu dữ liệu).
- Các hình **không có điểm nguyên chung**; nếu có điểm chung, điểm đó bị tính nhiều lần và không còn đều.

## 3. Kiến thức nền

### 3.1. Module `random` và `seed`

- `random.randint(lo, hi)`: số nguyên ngẫu nhiên trong `[lo, hi]` — **gồm cả hai đầu**.
- `random.randrange(n)`: số nguyên trong `[0, n - 1]` — **không gồm `n`**.
- `random.random()`: số thực trong `[0, 1)`.
- `random.seed(s)`: đặt "hạt giống"; **cùng seed ⇒ cùng dãy số**. Nhờ vậy kiểm thử ngẫu nhiên có thể **tái lập**.
- `random.Random(s)`: tạo bộ sinh riêng (không ảnh hưởng bộ sinh toàn cục) — rất tiện khi muốn so sánh nhiều cài đặt công bằng.

In [2]:
random.seed(2024)
print([random.randint(1, 6) for _ in range(8)])
random.seed(2024)                                   # đặt lại seed -> ra lại đúng dãy cũ
print([random.randint(1, 6) for _ in range(8)])

r1, r2 = random.Random(7), random.Random(7)
print([r1.randrange(10) for _ in range(6)], [r2.randrange(10) for _ in range(6)])
print("random() trong [0,1):", random.Random(1).random())

[4, 2, 6, 5, 3, 2, 6, 4]
[4, 2, 6, 5, 3, 2, 6, 4]
[5, 2, 6, 0, 1, 8] [5, 2, 6, 0, 1, 8]
random() trong [0,1): 0.13436424411240122


### 3.2. Tự dựng một bộ sinh số ngẫu nhiên có seed (để hiểu "ngẫu nhiên giả")

Máy tính không có ngẫu nhiên thật mà dùng **công thức xác định** (bộ sinh giả ngẫu nhiên). Ví dụ đơn giản nhất — **LCG** (linear congruential generator): `state = (a * state + c) % m`. Cùng `seed` ban đầu ⇒ cùng dãy. Ta dựng một lớp nhỏ có đủ `random()` và `randint()` để dùng được thay `random` trong các lớp của bài này.

> Đây chỉ để học. Trong code nộp LeetCode ta dùng `random` chuẩn (Mersenne Twister, chất lượng tốt hơn nhiều).

In [3]:
class LCG:
    """Bộ sinh giả ngẫu nhiên tự dựng (tham số kiểu glibc), chỉ để minh họa và test."""
    def __init__(self, seed):
        self.state = seed % (2 ** 31)

    def _next(self):
        self.state = (1103515245 * self.state + 12345) % (2 ** 31)
        return self.state

    def random(self):                      # [0, 1)
        return self._next() / 2 ** 31

    def randrange(self, n):                # [0, n-1]
        return int(self.random() * n)

    def randint(self, lo, hi):             # [lo, hi]
        return lo + self.randrange(hi - lo + 1)


g1, g2 = LCG(42), LCG(42)
print([g1.randint(1, 6) for _ in range(10)])
print([g2.randint(1, 6) for _ in range(10)])        # y hệt vì cùng seed

[4, 4, 3, 5, 3, 1, 3, 5, 4, 5]
[4, 4, 3, 5, 3, 1, 3, 5, 4, 5]


### 3.3. Cộng dồn (prefix sum) và tìm kiếm nhị phân `bisect`

Với dãy số lượng điểm `[16, 15]` ta tạo mảng **cộng dồn** `ends = [16, 31]`: phần tử `i` là tổng số điểm của các hình `0..i`. Một số `t` trong `[1, 31]` rơi vào hình đầu tiên có `ends[i] >= t`.

- `bisect_left(ends, t)`: vị trí **đầu tiên** `i` thỏa `ends[i] >= t`.
- `bisect_right(ends, t)`: vị trí đầu tiên thỏa `ends[i] > t`.

Tìm kiếm nhị phân trên mảng tăng mất `O(log k)`.

In [4]:
ends = [16, 31]
for t in [1, 15, 16, 17, 31]:
    print(f"t={t:>2}: bisect_left -> hình {bisect_left(ends, t)}")
print("bisect_right trên t = 0..30:", sorted({bisect_right(ends, t) for t in range(0, 31)}))
print("t=16 với bisect_right ->", bisect_right(ends, 16), "(khác bisect_left!)")

t= 1: bisect_left -> hình 0
t=15: bisect_left -> hình 0
t=16: bisect_left -> hình 0
t=17: bisect_left -> hình 1
t=31: bisect_left -> hình 1
bisect_right trên t = 0..30: [0, 1]
t=16 với bisect_right -> 1 (khác bisect_left!)


### 3.4. `divmod`, `//`, `%`: đổi một số nguyên thành tọa độ 2 chiều

Một lưới `w` cột có thể "trải phẳng" thành dãy `0, 1, ..., w*h - 1` theo từng hàng. Số `t` ứng với hàng `t // w`, cột `t % w`. `divmod(t, w)` trả cả hai. Đây là cách ánh xạ **1 số ngẫu nhiên → 1 điểm** cho Cách 4.

In [5]:
w, h = 4, 3
for t in [0, 1, 3, 4, 7, 11]:
    row, col = divmod(t, w)
    print(f"t={t:>2} -> hàng {row}, cột {col}  (kiểm tra: {row} * {w} + {col} = {row * w + col})")

t= 0 -> hàng 0, cột 0  (kiểm tra: 0 * 4 + 0 = 0)
t= 1 -> hàng 0, cột 1  (kiểm tra: 0 * 4 + 1 = 1)
t= 3 -> hàng 0, cột 3  (kiểm tra: 0 * 4 + 3 = 3)
t= 4 -> hàng 1, cột 0  (kiểm tra: 1 * 4 + 0 = 4)
t= 7 -> hàng 1, cột 3  (kiểm tra: 1 * 4 + 3 = 7)
t=11 -> hàng 2, cột 3  (kiểm tra: 2 * 4 + 3 = 11)


### 3.5. Chi-bình-phương (chi-square) — thước đo "có đều không"

Nếu có `N` mẫu và `K` kết quả đồng khả năng thì mỗi kết quả kỳ vọng xuất hiện `E = N / K` lần. Thống kê:

`chi2 = Σ (quan sát - E)^2 / E`

Nếu thật sự đều, `chi2` xấp xỉ **bậc tự do** `K - 1` (với độ lệch ngẫu nhiên). Nếu `chi2` **lớn hơn nhiều** bậc tự do thì phân bố **không đều**. Ta dùng ngưỡng xấp xỉ Wilson–Hilferty cho mức ý nghĩa rất nhỏ (`z = 3.72`, xác suất sai khoảng `1/10 000`) để chắc chắn không báo nhầm.

In [6]:
def chi_square(counts, total_expected_each):
    return sum((c - total_expected_each) ** 2 / total_expected_each for c in counts)


def chi2_critical(df, z=3.72):
    """Ngưỡng chi-bình-phương xấp xỉ (Wilson-Hilferty) cho bậc tự do df, z=3.72 ~ p=1e-4."""
    k = 2 / (9 * df)
    return df * (1 - k + z * k ** 0.5) ** 3


# Thử với xúc xắc 6 mặt: đều vs lệch
random.seed(1)
N = 6000
fair = [0] * 6
for _ in range(N):
    fair[random.randint(1, 6) - 1] += 1
loaded = [0] * 6
for _ in range(N):
    loaded[random.choice([1, 2, 3, 4, 5, 6, 6]) - 1] += 1       # mặt 6 hay ra hơn
print("đều  : chi2 =", round(chi_square(fair, N / 6), 2), " ngưỡng =", round(chi2_critical(5), 2))
print("lệch : chi2 =", round(chi_square(loaded, N / 6), 2), " ngưỡng =", round(chi2_critical(5), 2))

đều  : chi2 = 3.33  ngưỡng = 26.33
lệch : chi2 = 631.36  ngưỡng = 26.33


## 4. Ý tưởng nền tảng: tính tay

Với `rects = [[-2,-2,1,1], [2,2,4,6]]`: hình A có `4×4 = 16` điểm, hình B có `3×5 = 15` điểm, tổng `31`.

Muốn mỗi điểm có xác suất `1/31`:

- P(chọn hình A) = `16/31`, P(chọn hình B) = `15/31`;
- trong hình đã chọn, mỗi điểm có xác suất `1/16` (hoặc `1/15`) ⇒ tích: `16/31 × 1/16 = 1/31` ✓.

**Hai bước:** (1) chọn hình theo trọng số = số điểm; (2) chọn đều một điểm trong hình, tức chọn `u` đều trong `[a, x]` và `v` đều trong `[b, y]` **độc lập** (có `w` giá trị cho `u`, `h` giá trị cho `v`, mỗi cặp xác suất `1/(w·h)`).

Nếu ta **chọn hình đều nhau** (mỗi hình `1/2`) thì mỗi điểm trong A có xác suất `1/2 × 1/16 = 1/32` còn mỗi điểm trong B là `1/2 × 1/15 = 1/30` — **không đều** (ở ví dụ này lệch nhẹ, nhưng khi một hình rất nhỏ và một hình rất lớn thì lệch rất nhiều, xem mục 12.6).

In [7]:
rects = [[-2, -2, 1, 1], [2, 2, 4, 6]]
areas = [(x - a + 1) * (y - b + 1) for a, b, x, y in rects]
total = sum(areas)
print("areas =", areas, "total =", total)
for i, ar in enumerate(areas):
    print(f"hình {i}: P(chọn hình) = {ar}/{total}, P(1 điểm trong hình) = 1/{ar}, tích = {ar / total * (1 / ar):.5f} = 1/{total}")
print("chọn hình đều thì điểm trong A:", 1 / 2 * 1 / areas[0], " trong B:", 1 / 2 * 1 / areas[1])

areas = [16, 15] total = 31
hình 0: P(chọn hình) = 16/31, P(1 điểm trong hình) = 1/16, tích = 0.03226 = 1/31
hình 1: P(chọn hình) = 15/31, P(1 điểm trong hình) = 1/15, tích = 0.03226 = 1/31
chọn hình đều thì điểm trong A: 0.03125  trong B: 0.03333333333333333


## 5. Cách 1 — Brute Force: liệt kê mọi điểm rồi `choice`

**Ý tưởng.** Trong `__init__`, duyệt từng hình, thêm **mọi điểm nguyên** vào một list `points`. `pick()` trả `random.choice(points)` — hiển nhiên đồng khả năng vì list chứa mỗi điểm đúng một lần.

**Code demo:** (các lớp demo nhận một bộ sinh `rng` để so sánh công bằng; mặc định dùng module `random`).

In [8]:
class PickBrute:
    def __init__(self, rects, rng=random):
        self.rng = rng
        self.points = []
        for a, b, x, y in rects:
            for u in range(a, x + 1):
                for v in range(b, y + 1):
                    self.points.append([u, v])

    def pick(self):
        return self.rng.choice(self.points)


pb = PickBrute(rects_demo, random.Random(5))
print("số điểm:", len(pb.points))
print([pb.pick() for _ in range(5)])

số điểm: 31
[[2, 5], [0, -2], [3, 4], [0, 1], [3, 6]]


**Code nộp LeetCode (Cách 1):**

In [9]:
class SolutionBrute:
    def __init__(self, rects: List[List[int]], rng=random):
        self.rng = rng
        self.points: List[List[int]] = []
        for a, b, x, y in rects:
            for u in range(a, x + 1):
                for v in range(b, y + 1):
                    self.points.append([u, v])

    def pick(self) -> List[int]:
        return self.rng.choice(self.points)


s = SolutionBrute(rects_demo, random.Random(1))
print(s.pick(), s.pick())

[-1, -2] [2, 4]


**Độ phức tạp.** Khởi tạo `O(T)` thời gian **và bộ nhớ**, với `T` = tổng số điểm; `pick` `O(1)`.

**Nút thắt.** `T` có thể tới `~4 × 10^8` (100 hình, mỗi hình `2001²` điểm). Một list 400 triệu điểm cần hàng chục GB — **Memory Limit Exceeded**. Ta cần **không bao giờ liệt kê điểm**, chỉ làm việc với **số lượng** điểm của từng hình.

In [10]:
big_rects = [[i * 3000, 0, i * 3000 + 2000, 2000] for i in range(100)]      # 100 hình, mỗi hình 2001 x 2001 điểm
T = sum((x - a + 1) * (y - b + 1) for a, b, x, y in big_rects)
print("tổng số điểm:", f"{T:,}")
print("ước lượng bộ nhớ nếu mỗi điểm là list 2 số (~72 byte):", f"{T * 72 / 1e9:.0f} GB")

tổng số điểm: 400,400,100
ước lượng bộ nhớ nếu mỗi điểm là list 2 số (~72 byte): 29 GB


## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — Không cần liệt kê điểm.** Điểm nằm trong hình `i` được xác định bởi `(u, v)` với `u ∈ [a, x]`, `v ∈ [b, y]`. Chọn đều `u` và `v` độc lập cho điểm đều trong hình.

**Bước 2 — Chỉ cần chọn hình theo trọng số.** Trọng số của hình `i` là `cnt_i = (x_i - a_i + 1)(y_i - b_i + 1)`. Gọi `T = Σ cnt_i`. Chọn hình `i` với xác suất `cnt_i / T`.

**Bước 3 — Chọn theo trọng số như thế nào?** Trải `T` điểm lên một "thanh" độ dài `T` gồm các đoạn nối nhau độ dài `cnt_0, cnt_1, ...`. Chọn **một số nguyên đều** `t ∈ [1, T]`; đoạn chứa `t` là hình được chọn. Có nhiều cách tìm đoạn:

- quét tuyến tính `O(k)` (Cách 2);
- mảng cộng dồn + **tìm nhị phân** `O(log k)` (Cách 3);
- dùng luôn `t` để suy ra tọa độ với `divmod` (Cách 4);
- hoặc **không** cộng dồn trước mà chọn "trực tuyến" bằng **reservoir sampling có trọng số** (Cách 5).

## 7. Các cách tiếp theo

### Cách 2 — Chọn hình theo trọng số bằng quét tuyến tính

**Giải thích từng đoạn.**

1. Tính `cnt[i]` cho mỗi hình và `T = sum(cnt)` trong `__init__`.
2. `pick`: `t = randint(1, T)` (đều trong `[1, T]`).
3. Duyệt `i` từ `0`: nếu `t <= cnt[i]` thì chọn hình `i`; nếu không trừ `t -= cnt[i]` và sang hình sau. (Giống "rơi vào đoạn nào".)
4. Chọn `u = randint(a, x)`, `v = randint(b, y)`.

In [11]:
class PickScan:
    def __init__(self, rects, rng=random):
        self.rng = rng
        self.rects = rects
        self.cnt = [(x - a + 1) * (y - b + 1) for a, b, x, y in rects]
        self.total = sum(self.cnt)

    def pick(self):
        t = self.rng.randint(1, self.total)
        for i, c in enumerate(self.cnt):
            if t <= c:
                break
            t -= c
        a, b, x, y = self.rects[i]
        return [self.rng.randint(a, x), self.rng.randint(b, y)]


ps = PickScan(rects_demo, random.Random(3))
print([ps.pick() for _ in range(5)])

[[-1, 0], [4, 5], [4, 2], [2, 5], [-1, -1]]


In [12]:
class SolutionScan:
    def __init__(self, rects: List[List[int]], rng=random):
        self.rng = rng
        self.rects = rects
        self.cnt = [(x - a + 1) * (y - b + 1) for a, b, x, y in rects]
        self.total = sum(self.cnt)

    def pick(self) -> List[int]:
        t = self.rng.randint(1, self.total)
        i = 0
        while t > self.cnt[i]:
            t -= self.cnt[i]
            i += 1
        a, b, x, y = self.rects[i]
        return [self.rng.randint(a, x), self.rng.randint(b, y)]


s = SolutionScan(rects_demo, random.Random(2))
print(s.pick(), s.pick())

[2, 2] [0, -1]


**Độ phức tạp.** `__init__`: `O(k)` với `k = len(rects)`. `pick`: `O(k)` (quét). Bộ nhớ `O(k)`. Với `k <= 100` và `10^4` lần gọi: khoảng `10^6` thao tác — đủ nhanh, nhưng ta có thể làm tốt hơn.

### Cách 3 — Cộng dồn + tìm kiếm nhị phân (chốt nộp bài)

**Giải thích từng đoạn.**

1. `ends[i]` = tổng số điểm của các hình `0..i` (mảng cộng dồn, **tăng chặt** vì mỗi hình có ít nhất `2 × 2 = 4` điểm).
2. `t = randint(1, ends[-1])`.
3. `i = bisect_left(ends, t)`: hình đầu tiên có `ends[i] >= t` — đúng hình chứa `t`.
4. Chọn `u`, `v` đều trong hình `i`.

`pick` mất `O(log k)` thay vì `O(k)`.

In [13]:
class PickBisect:
    def __init__(self, rects, rng=random):
        self.rng = rng
        self.rects = rects
        self.ends = []
        total = 0
        for a, b, x, y in rects:
            total += (x - a + 1) * (y - b + 1)
            self.ends.append(total)

    def pick(self):
        t = self.rng.randint(1, self.ends[-1])
        i = bisect_left(self.ends, t)
        a, b, x, y = self.rects[i]
        return [self.rng.randint(a, x), self.rng.randint(b, y)]


pq = PickBisect(rects_demo, random.Random(4))
print(pq.ends, [pq.pick() for _ in range(5)])

[16, 31] [[0, -2], [3, 5], [-2, -2], [1, 0], [2, 3]]


**Code nộp LeetCode (Cách 3, dùng module `random` toàn cục như LeetCode mong đợi):**

In [14]:
class Solution:
    def __init__(self, rects: List[List[int]]):
        self.rects = rects
        self.ends: List[int] = []                    # ends[i] = số điểm của các hình 0..i
        total = 0
        for a, b, x, y in rects:
            total += (x - a + 1) * (y - b + 1)       # biên được tính: +1 ở mỗi chiều
            self.ends.append(total)

    def pick(self) -> List[int]:
        t = random.randint(1, self.ends[-1])         # đều trong [1, T]
        i = bisect_left(self.ends, t)                # hình chứa t
        a, b, x, y = self.rects[i]
        return [random.randint(a, x), random.randint(b, y)]


random.seed(1)
obj = Solution(rects_demo)
print([obj.pick() for _ in range(5)])

[[-2, 0], [1, 1], [1, -1], [1, -2], [3, 5]]


**Độ phức tạp.** `__init__`: `O(k)`. `pick`: `O(log k)` cho tìm nhị phân + `O(1)` cho hai lần `randint`. Bộ nhớ `O(k)`.

### Cách 4 — Một số ngẫu nhiên duy nhất rồi `divmod`

**Ý tưởng khác.** Thay vì gọi `randint` ba lần, ta gọi `randrange(T)` **một lần** để chọn **điểm thứ `t`** trong toàn bộ `T` điểm, rồi **giải mã** `t` thành điểm:

1. `i = bisect_right(ends, t)` (với `t ∈ [0, T-1]`, hình đầu tiên có `ends[i] > t`);
2. `offset = t - (ends[i] - cnt[i])` — chỉ số điểm bên trong hình `i`, trong `[0, cnt[i] - 1]`;
3. `row, col = divmod(offset, w)` với `w = x - a + 1` (số cột) ⇒ `u = a + col`, `v = b + row`.

Phép giải mã này là **song ánh** từ `[0, T)` lên tập điểm, nên mỗi điểm đúng một `t` ⇒ đều. Ta cũng có thể "đánh số" điểm theo cách bất kỳ — chỉ cần song ánh.

In [15]:
class PickDivmod:
    def __init__(self, rects, rng=random):
        self.rng = rng
        self.rects = rects
        self.ends = []
        total = 0
        for a, b, x, y in rects:
            total += (x - a + 1) * (y - b + 1)
            self.ends.append(total)

    def decode(self, t):
        """Điểm thứ t (0-based) trong toàn bộ T điểm."""
        i = bisect_right(self.ends, t)
        a, b, x, y = self.rects[i]
        start = self.ends[i] - (x - a + 1) * (y - b + 1)
        row, col = divmod(t - start, x - a + 1)
        return [a + col, b + row]

    def pick(self):
        return self.decode(self.rng.randrange(self.ends[-1]))


pd = PickDivmod(rects_demo)
print([pd.decode(t) for t in range(0, 31, 5)])
all_points = [pd.decode(t) for t in range(pd.ends[-1])]
print("31 điểm đều khác nhau:", len({tuple(p) for p in all_points}) == 31)

[[-2, -2], [-1, -1], [0, 0], [1, 1], [3, 3], [2, 5], [4, 6]]
31 điểm đều khác nhau: True


In [16]:
class SolutionDivmod:
    def __init__(self, rects: List[List[int]], rng=random):
        self.rng = rng
        self.rects = rects
        self.ends: List[int] = []
        total = 0
        for a, b, x, y in rects:
            total += (x - a + 1) * (y - b + 1)
            self.ends.append(total)

    def pick(self) -> List[int]:
        t = self.rng.randrange(self.ends[-1])        # điểm thứ t, t trong [0, T-1]
        i = bisect_right(self.ends, t)
        a, b, x, y = self.rects[i]
        start = self.ends[i] - (x - a + 1) * (y - b + 1)
        row, col = divmod(t - start, x - a + 1)
        return [a + col, b + row]


s = SolutionDivmod(rects_demo, random.Random(9))
print(s.pick(), s.pick())

[0, 1] [2, 3]


**Độ phức tạp.** `__init__` `O(k)`; `pick` `O(log k)` với **một** lần gọi bộ sinh số ngẫu nhiên. Bộ nhớ `O(k)`.

### Cách 5 — Reservoir sampling có trọng số (không cần cộng dồn trước)

**Ý tưởng.** Reservoir sampling chọn một phần tử đều từ dòng dữ liệu chưa biết độ dài. Bản **có trọng số** để chọn hình `i` với xác suất `cnt_i / T` chỉ bằng **một lượt quét**: giữ `running` = tổng trọng số đã thấy; với hình `i` mới, **thay** hình đang giữ bằng hình `i` với xác suất `cnt_i / running`.

**Vì sao đúng?** Xác suất hình `i` được giữ đến cuối = (xác suất `i` được chọn khi gặp) × (xác suất không bị thay bởi `i+1, ..., k-1`):

`cnt_i/W_i × (W_i/W_{i+1}) × (W_{i+1}/W_{i+2}) × ... × (W_{k-2}/W_{k-1}) = cnt_i / W_{k-1} = cnt_i / T`

(với `W_i = cnt_0 + ... + cnt_i`; các thừa số triệt tiêu tuần tự.) Nên đúng tỉ lệ cần thiết.

In [17]:
class PickReservoir:
    def __init__(self, rects, rng=random):
        self.rng = rng
        self.rects = rects
        self.cnt = [(x - a + 1) * (y - b + 1) for a, b, x, y in rects]

    def pick(self):
        running = 0
        chosen = 0
        for i, c in enumerate(self.cnt):
            running += c
            if self.rng.random() * running < c:      # xác suất c / running
                chosen = i
        a, b, x, y = self.rects[chosen]
        return [self.rng.randint(a, x), self.rng.randint(b, y)]


pr = PickReservoir(rects_demo, random.Random(6))
print([pr.pick() for _ in range(5)])

[[1, 0], [4, 5], [0, -2], [1, -2], [0, -2]]


In [18]:
class SolutionReservoir:
    def __init__(self, rects: List[List[int]], rng=random):
        self.rng = rng
        self.rects = rects
        self.cnt = [(x - a + 1) * (y - b + 1) for a, b, x, y in rects]

    def pick(self) -> List[int]:
        running = 0
        chosen = 0
        for i, c in enumerate(self.cnt):
            running += c
            if self.rng.random() * running < c:      # giữ hình i với xác suất c / running
                chosen = i
        a, b, x, y = self.rects[chosen]
        return [self.rng.randint(a, x), self.rng.randint(b, y)]


s = SolutionReservoir(rects_demo, random.Random(8))
print(s.pick(), s.pick())

[-1, -1] [2, 6]


**Độ phức tạp.** `pick` quét `k` hình ⇒ `O(k)` thời gian, `k` lần gọi `random()`; bộ nhớ `O(k)` (hoặc `O(1)` nếu tính `cnt` khi quét). Cách này hữu ích khi dữ liệu là **dòng** hoặc thay đổi liên tục (không giữ được cộng dồn); ở bài này nó chậm hơn Cách 3. Lưu ý: `rng.random() * running < c` dùng số thực nên độ chính xác phụ thuộc kiểu `float` 53 bit — vẫn dư dùng cho `T ≤ 4×10^8`.

## 8. TRACE từng bước

Trace Cách 3 với `rects = [[-2,-2,1,1], [2,2,4,6]]` và một bộ sinh `LCG(2024)` tự dựng (kết quả lặp lại được).

| bước | giá trị | ý nghĩa |
|---|---|---|
| 1 | `cnt = [16, 15]` | số điểm từng hình |
| 2 | `ends = [16, 31]` | cộng dồn |
| 3 | `t = randint(1, 31)` | chọn "điểm thứ t" trong 31 điểm |
| 4 | `i = bisect_left(ends, t)` | `t <= 16` ⇒ hình 0; `t >= 17` ⇒ hình 1 |
| 5 | `u = randint(a, x)`, `v = randint(b, y)` | tọa độ đều trong hình đã chọn |

Hàm `trace_pick` in từng bước cho vài lần `pick`; `trace_decode` in bảng giải mã của Cách 4.

In [19]:
def trace_pick(rects, n_picks, seed):
    rng = LCG(seed)
    cnt = [(x - a + 1) * (y - b + 1) for a, b, x, y in rects]
    ends, run = [], 0
    for c in cnt:
        run += c
        ends.append(run)
    print("rects =", rects)
    print("cnt   =", cnt, " ends =", ends, " T =", ends[-1])
    for k in range(1, n_picks + 1):
        t = rng.randint(1, ends[-1])
        i = bisect_left(ends, t)
        a, b, x, y = rects[i]
        u, v = rng.randint(a, x), rng.randint(b, y)
        print(f"pick #{k}: t={t:>2} -> bisect_left={i} (hình {rects[i]}), u in [{a},{x}] -> {u}, v in [{b},{y}] -> {v}  => [{u}, {v}]")


trace_pick(rects_demo, 6, 2024)

rects = [[-2, -2, 1, 1], [2, 2, 4, 6]]
cnt   = [16, 15]  ends = [16, 31]  T = 31
pick #1: t= 2 -> bisect_left=0 (hình [-2, -2, 1, 1]), u in [-2,1] -> -1, v in [-2,1] -> -2  => [-1, -2]
pick #2: t=14 -> bisect_left=0 (hình [-2, -2, 1, 1]), u in [-2,1] -> -2, v in [-2,1] -> 0  => [-2, 0]
pick #3: t=23 -> bisect_left=1 (hình [2, 2, 4, 6]), u in [2,4] -> 3, v in [2,6] -> 2  => [3, 2]
pick #4: t= 7 -> bisect_left=0 (hình [-2, -2, 1, 1]), u in [-2,1] -> 1, v in [-2,1] -> -1  => [1, -1]
pick #5: t=22 -> bisect_left=1 (hình [2, 2, 4, 6]), u in [2,4] -> 3, v in [2,6] -> 5  => [3, 5]
pick #6: t= 6 -> bisect_left=0 (hình [-2, -2, 1, 1]), u in [-2,1] -> -1, v in [-2,1] -> 1  => [-1, 1]


In [20]:
def trace_decode(rects):
    ends, run = [], 0
    for a, b, x, y in rects:
        run += (x - a + 1) * (y - b + 1)
        ends.append(run)
    print("ends =", ends)
    print(f"{'t':>2} | {'i':>1} | {'start':>5} | {'offset':>6} | {'w':>1} | (row, col) | điểm")
    for t in range(ends[-1]):
        i = bisect_right(ends, t)
        a, b, x, y = rects[i]
        w = x - a + 1
        start = ends[i] - w * (y - b + 1)
        row, col = divmod(t - start, w)
        if t in (0, 1, 3, 4, 15, 16, 17, 19, 30):
            print(f"{t:>2} | {i:>1} | {start:>5} | {t - start:>6} | {w:>1} | ({row}, {col})     | [{a + col}, {b + row}]")


trace_decode(rects_demo)

ends = [16, 31]
 t | i | start | offset | w | (row, col) | điểm
 0 | 0 |     0 |      0 | 4 | (0, 0)     | [-2, -2]
 1 | 0 |     0 |      1 | 4 | (0, 1)     | [-1, -2]
 3 | 0 |     0 |      3 | 4 | (0, 3)     | [1, -2]
 4 | 0 |     0 |      4 | 4 | (1, 0)     | [-2, -1]
15 | 0 |     0 |     15 | 4 | (3, 3)     | [1, 1]
16 | 1 |    16 |      0 | 3 | (0, 0)     | [2, 2]
17 | 1 |    16 |      1 | 3 | (0, 1)     | [3, 2]
19 | 1 |    16 |      3 | 3 | (1, 0)     | [2, 3]
30 | 1 |    16 |     14 | 3 | (4, 2)     | [4, 6]


## 9. Code nộp LeetCode

Ta nộp Cách 3 (cộng dồn + `bisect_left`). Lưu ý phải có `import random` và `from bisect import bisect_left` ở đầu file (trên LeetCode thường đã có sẵn `random`, `bisect`; nếu muốn chắc, tự thêm import).

In [21]:
import random
from bisect import bisect_left
from typing import List


class Solution:
    def __init__(self, rects: List[List[int]]):
        self.rects = rects
        self.ends: List[int] = []
        total = 0
        for a, b, x, y in rects:
            total += (x - a + 1) * (y - b + 1)
            self.ends.append(total)

    def pick(self) -> List[int]:
        t = random.randint(1, self.ends[-1])
        i = bisect_left(self.ends, t)
        a, b, x, y = self.rects[i]
        return [random.randint(a, x), random.randint(b, y)]


# Your Solution object will be instantiated and called as such:
# obj = Solution(rects)
# param_1 = obj.pick()
random.seed(497)
obj = Solution([[-2, -2, 1, 1], [2, 2, 4, 6]])
print([obj.pick() for _ in range(5)])

[[-1, 0], [0, -1], [2, 2], [2, 4], [3, 4]]


## 10. Vì sao đúng

**Đều trên mọi điểm.** Gọi `T = Σ cnt_i`. Với một điểm `p` thuộc hình `i`:

`P(trả p) = P(chọn hình i) × P(chọn u đúng) × P(chọn v đúng) = (cnt_i / T) × (1/w_i) × (1/h_i) = (cnt_i / T) × (1 / cnt_i) = 1 / T`

vì `cnt_i = w_i · h_i`. Mọi điểm có cùng xác suất `1/T`.

**`P(chọn hình i) = cnt_i / T`.** `t = randint(1, T)` đều; các giá trị `t` mà `bisect_left(ends, t) == i` là `ends[i-1] + 1, ..., ends[i]` — đúng `cnt_i` số ⇒ xác suất `cnt_i / T`.

**Hình chữ nhật không chồng lấn** nên mỗi điểm thuộc đúng một hình, không bị tính trùng trọng số.

**Điểm trả về nằm trong hình.** `u ∈ [a, x]`, `v ∈ [b, y]` (cả hai đầu đều được `randint` gồm) ⇒ điểm nguyên thuộc hình chữ nhật kể cả biên.

**Cách 4.** `decode` là song ánh `[0, T) → điểm`: `i` xác định hình, `offset ∈ [0, cnt_i)` và `(row, col) = divmod(offset, w)` có `row ∈ [0, h)`, `col ∈ [0, w)` — mỗi cặp ứng với đúng một `offset`.

**Cách 5.** Chứng minh tích triệt tiêu ở mục 7.

## 11. Time / Space Complexity

Gọi `k = len(rects) <= 100`, `T` = tổng số điểm nguyên.

| Cách | `__init__` | `pick` | Space |
|---|---|---|---|
| 1. Liệt kê điểm | `O(T)` | `O(1)` | `O(T)` (không khả thi) |
| 2. Quét tuyến tính | `O(k)` | `O(k)` | `O(k)` |
| 3. Cộng dồn + bisect | `O(k)` | `O(log k)` | `O(k)` |
| 4. Một số + divmod | `O(k)` | `O(log k)` | `O(k)` |
| 5. Reservoir có trọng số | `O(k)` | `O(k)` | `O(k)` |

Với `10^4` lần `pick` và `k <= 100`, mọi cách trừ Cách 1 đều chạy rất nhanh; Cách 3 là điểm cân bằng tốt nhất.

## 12. Bộ test

Vì bài **ngẫu nhiên**, ta không so đáp án cứng. Ta kiểm tra:

1. **Tính hợp lệ:** mọi điểm trả về nằm trong **đúng một** hình chữ nhật (đếm số hình chứa điểm `== 1`).
2. **Tính tái lập:** cùng seed ⇒ cùng dãy.
3. **Phân bố đều:** số lần mỗi điểm xuất hiện tuân theo chi-bình-phương (với seed cố định để kết quả lặp lại được; ngưỡng đủ rộng để không báo nhầm nhưng đủ chặt để bắt cài đặt sai).
4. **Phân bố theo hình:** tỉ lệ số lần rơi vào mỗi hình ≈ `cnt_i / T`.
5. **Phản ví dụ:** cài đặt sai (chọn hình đều; quên `+1`) bị thống kê phát hiện.

Ngoài ra có hàm `run_ops` mô phỏng cách LeetCode gọi chuỗi lệnh `["Solution","pick",...]`.

In [22]:
def contains(rect, p):
    a, b, x, y = rect
    return a <= p[0] <= x and b <= p[1] <= y


def owners(rects, p):
    return sum(contains(r, p) for r in rects)


def count_points(rects):
    return sum((x - a + 1) * (y - b + 1) for a, b, x, y in rects)


def run_ops(cls, ops, args):
    """Mô phỏng LeetCode: ops = ['Solution','pick',...], args = [[rects], [], ...]."""
    obj = None
    out = []
    for op, arg in zip(ops, args):
        if op == "Solution":
            obj = cls(*arg)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*arg))
    return out


def make_disjoint_rects(rng, k, base=(-30, -30)):
    """Sinh k hình không có điểm nguyên chung: mỗi hình trong một ô lưới 12x12 với khoảng hở."""
    cells = [(i, j) for i in range(5) for j in range(5)]
    rng.shuffle(cells)
    rects = []
    for ci, cj in cells[:k]:
        a = base[0] + ci * 12 + rng.randint(0, 3)
        b = base[1] + cj * 12 + rng.randint(0, 3)
        x = a + rng.randint(1, 6)
        y = b + rng.randint(1, 6)
        rects.append([a, b, x, y])
    return rects


ops = ["Solution", "pick", "pick", "pick", "pick", "pick"]
args = [[rects_demo], [], [], [], [], []]
random.seed(497)
res = run_ops(Solution, ops, args)
print(res)
assert res[0] is None and all(owners(rects_demo, p) == 1 for p in res[1:])

[None, [-1, 0], [0, -1], [2, 2], [2, 4], [3, 4]]


In [23]:
# 12.1 Tính hợp lệ: điểm nằm trong đúng một hình (mọi cài đặt), nhiều bộ rects ngẫu nhiên có seed
def factories():
    return {
        "Solution (global random)": lambda rects, rng: Solution(rects),
        "Brute": lambda rects, rng: SolutionBrute(rects, rng),
        "Scan": lambda rects, rng: SolutionScan(rects, rng),
        "Divmod": lambda rects, rng: SolutionDivmod(rects, rng),
        "Reservoir": lambda rects, rng: SolutionReservoir(rects, rng),
    }


gen = random.Random(497)
for trial in range(40):
    k = gen.randint(1, 8)
    rects = make_disjoint_rects(gen, k)
    # kiểm tra các hình thật sự không chung điểm nguyên: tổng số điểm = số điểm phân biệt
    pts = {(u, v) for a, b, x, y in rects for u in range(a, x + 1) for v in range(b, y + 1)}
    assert len(pts) == count_points(rects)
    for name, make in factories().items():
        random.seed(trial)
        obj = make(rects, random.Random(trial))
        for _ in range(200):
            p = obj.pick()
            assert len(p) == 2 and owners(rects, p) == 1, (name, rects, p)
print("OK: 40 bộ rects x", len(factories()), "cài đặt x 200 lần pick -> luôn nằm trong đúng một hình")

OK: 40 bộ rects x 5 cài đặt x 200 lần pick -> luôn nằm trong đúng một hình


In [24]:
# 12.2 Cực trị tọa độ: hình ở sát +-10^9 và hình có kích thước tối đa 2001 x 2001
extreme = [
    [-10 ** 9, -10 ** 9, -10 ** 9 + 2000, -10 ** 9 + 2000],
    [10 ** 9 - 2000, 10 ** 9 - 2000, 10 ** 9, 10 ** 9],
    [0, 0, 1, 1],
]
random.seed(1)
obj = Solution(extreme)
for _ in range(3000):
    p = obj.pick()
    assert owners(extreme, p) == 1, p
print("tổng số điểm:", f"{count_points(extreme):,}", "-> OK")

# hình nhỏ nhất: a < x nên mỗi chiều có ít nhất 2 điểm
one = [[5, 5, 6, 6]]
random.seed(2)
obj = Solution(one)
seen = {tuple(obj.pick()) for _ in range(400)}
assert seen == {(5, 5), (5, 6), (6, 5), (6, 6)}, seen
print("hình 2x2: đủ 4 điểm, kể cả biên:", sorted(seen))

tổng số điểm: 8,008,006 -> OK
hình 2x2: đủ 4 điểm, kể cả biên: [(5, 5), (5, 6), (6, 5), (6, 6)]


In [25]:
# 12.3 Tính tái lập: cùng seed -> cùng dãy; khác seed -> (gần như chắc chắn) khác dãy
def sequence(seed, n=20):
    random.seed(seed)
    o = Solution(rects_demo)
    return [tuple(o.pick()) for _ in range(n)]


assert sequence(11) == sequence(11)
assert sequence(11) != sequence(12)
print("tái lập OK:", sequence(11)[:4])

tái lập OK: [(1, 1), (4, 3), (1, -1), (1, 0)]


In [26]:
# 12.4 Phân bố đều theo từng điểm bằng chi-bình-phương
def check_uniform(make, rects, n_samples, seed, label=""):
    rng = random.Random(seed)
    random.seed(seed)                              # cho Solution dùng random toàn cục
    obj = make(rects, rng)
    pts = [(u, v) for a, b, x, y in rects for u in range(a, x + 1) for v in range(b, y + 1)]
    counts = {p: 0 for p in pts}
    for _ in range(n_samples):
        p = tuple(obj.pick())
        counts[p] += 1                              # KeyError nếu ra ngoài tập điểm hợp lệ
    expected = n_samples / len(pts)
    stat = chi_square(counts.values(), expected)
    crit = chi2_critical(len(pts) - 1)
    return stat, crit, min(counts.values()), max(counts.values())


N = 31000                                           # kỳ vọng 1000 lần mỗi điểm
for name, make in factories().items():
    stat, crit, lo, hi = check_uniform(make, rects_demo, N, seed=497)
    status = "ĐỀU" if stat < crit else "LỆCH"
    print(f"{name:<26} chi2 = {stat:6.2f} (ngưỡng {crit:5.1f}) min={lo} max={hi} -> {status}")
    assert stat < crit, name

Solution (global random)   chi2 =  21.26 (ngưỡng  67.9) min=958 max=1086 -> ĐỀU


Brute                      chi2 =  17.14 (ngưỡng  67.9) min=956 max=1063 -> ĐỀU
Scan                       chi2 =  21.26 (ngưỡng  67.9) min=958 max=1086 -> ĐỀU


Divmod                     chi2 =  17.14 (ngưỡng  67.9) min=956 max=1063 -> ĐỀU
Reservoir                  chi2 =  16.31 (ngưỡng  67.9) min=951 max=1053 -> ĐỀU


In [27]:
# 12.5 Phân bố theo hình: tỉ lệ ~ cnt_i / T (kiểm tra trực tiếp ý chọn theo trọng số)
rects3 = [[0, 0, 1, 1], [3, 0, 7, 4], [10, 10, 13, 20]]          # 4, 25, 44 điểm
cnt = [(x - a + 1) * (y - b + 1) for a, b, x, y in rects3]
T = sum(cnt)
N = 73000
random.seed(1234)
obj = Solution(rects3)
hit = [0] * 3
for _ in range(N):
    p = obj.pick()
    hit[[contains(r, p) for r in rects3].index(True)] += 1
print("cnt =", cnt, " T =", T)
for i in range(3):
    exp = N * cnt[i] / T
    print(f"hình {i}: quan sát {hit[i]:>6}, kỳ vọng {exp:8.1f}, lệch {abs(hit[i] - exp) / exp * 100:.2f}%")
stat = sum((hit[i] - N * cnt[i] / T) ** 2 / (N * cnt[i] / T) for i in range(3))
assert stat < chi2_critical(2), stat
print("chi2 =", round(stat, 2), "< ngưỡng", round(chi2_critical(2), 2), "-> OK")

cnt = [4, 25, 44]  T = 73
hình 0: quan sát   4049, kỳ vọng   4000.0, lệch 1.23%
hình 1: quan sát  25152, kỳ vọng  25000.0, lệch 0.61%
hình 2: quan sát  43799, kỳ vọng  44000.0, lệch 0.46%
chi2 = 2.44 < ngưỡng 19.3 -> OK


In [28]:
# 12.6 Cài đặt SAI bị thống kê bắt được (phản ví dụ)
class WrongUniformRect:
    """SAI: chọn hình đều nhau (mỗi hình 1/k) rồi điểm đều trong hình."""
    def __init__(self, rects, rng):
        self.rects, self.rng = rects, rng

    def pick(self):
        a, b, x, y = self.rng.choice(self.rects)
        return [self.rng.randint(a, x), self.rng.randint(b, y)]


class WrongNoPlusOne:
    """SAI: trọng số (x-a)*(y-b) thiếu +1 ở mỗi chiều (quên biên) nhưng vẫn chọn tọa độ gồm biên."""
    def __init__(self, rects, rng):
        self.rects, self.rng = rects, rng
        self.w = [(x - a) * (y - b) for a, b, x, y in rects]

    def pick(self):
        i = self.rng.choices(range(len(self.rects)), weights=self.w)[0]
        a, b, x, y = self.rects[i]
        return [self.rng.randint(a, x), self.rng.randint(b, y)]


rects_skew = [[0, 0, 1, 1], [3, 3, 9, 9]]                 # 4 điểm và 49 điểm: chênh lệch lớn để lỗi lộ rõ
for name, cls in [("chọn hình đều", WrongUniformRect), ("quên +1 ở diện tích", WrongNoPlusOne)]:
    stat, crit, lo, hi = check_uniform(lambda r, g, c=cls: c(r, g), rects_skew, 53000, seed=497)
    print(f"{name:<22} chi2 = {stat:9.2f} (ngưỡng {crit:5.1f}) min={lo} max={hi} -> {'LỆCH (bị bắt)' if stat >= crit else 'không phát hiện'}")
    assert stat >= crit
stat, crit, lo, hi = check_uniform(lambda r, g: Solution(r), rects_skew, 53000, seed=497)
print(f"{'Solution đúng':<22} chi2 = {stat:9.2f} (ngưỡng {crit:5.1f}) min={lo} max={hi} -> ĐỀU")
assert stat < crit
print("OK: cả hai cài đặt sai đều bị phát hiện")

chọn hình đều          chi2 = 139328.59 (ngưỡng  98.9) min=480 max=6759 -> LỆCH (bị bắt)
quên +1 ở diện tích    chi2 =   1913.06 (ngưỡng  98.9) min=325 max=1127 -> LỆCH (bị bắt)


Solution đúng          chi2 =     42.78 (ngưỡng  98.9) min=942 max=1058 -> ĐỀU
OK: cả hai cài đặt sai đều bị phát hiện


In [29]:
# 12.7 Bộ sinh tự dựng LCG: lớp demo vẫn đều (không dùng random của Python)
for name, cls in [("Scan", SolutionScan), ("Divmod", SolutionDivmod), ("Reservoir", SolutionReservoir)]:
    stat, crit, lo, hi = check_uniform(lambda r, g, c=cls: c(r, LCG(31337)), rects_demo, 31000, seed=1)
    print(f"{name:<10} + LCG: chi2 = {stat:6.2f} (ngưỡng {crit:5.1f}) min={lo} max={hi}")
    assert stat < crit, name
print("OK")

Scan       + LCG: chi2 =  35.51 (ngưỡng  67.9) min=932 max=1071
Divmod     + LCG: chi2 =  27.46 (ngưỡng  67.9) min=946 max=1063


Reservoir  + LCG: chi2 =  18.99 (ngưỡng  67.9) min=946 max=1042
OK


In [30]:
# 12.8 Tốc độ: 10^4 lần pick trên 100 hình lớn nhất; Cách 1 không khả thi nên bỏ qua
random.seed(99)
big_rects = [[i * 3000, -1000, i * 3000 + 2000, 1000] for i in range(100)]
for name, make in [("bisect (Solution)", lambda r: Solution(r)), ("scan", lambda r: SolutionScan(r)),
                   ("divmod", lambda r: SolutionDivmod(r)), ("reservoir", lambda r: SolutionReservoir(r))]:
    obj = make(big_rects)
    t0 = time.perf_counter()
    for _ in range(10_000):
        p = obj.pick()
    dt = time.perf_counter() - t0
    assert owners(big_rects, p) == 1
    print(f"{name:<18}: 10^4 pick trong {dt:.3f}s")

bisect (Solution) : 10^4 pick trong 0.013s
scan              : 10^4 pick trong 0.036s
divmod            : 10^4 pick trong 0.007s


reservoir         : 10^4 pick trong 0.092s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên `+1` khi đếm điểm nguyên (biên cũng là điểm)

In [31]:
rect = [0, 0, 2, 1]                       # hoành độ 0,1,2 (3 giá trị), tung độ 0,1 (2 giá trị) -> 6 điểm
a, b, x, y = rect
print("đếm sai (x-a)*(y-b)         =", (x - a) * (y - b))
print("đếm đúng (x-a+1)*(y-b+1)    =", (x - a + 1) * (y - b + 1))
print("liệt kê:", [[u, v] for u in range(a, x + 1) for v in range(b, y + 1)])

đếm sai (x-a)*(y-b)         = 2
đếm đúng (x-a+1)*(y-b+1)    = 6
liệt kê: [[0, 0], [0, 1], [1, 0], [1, 1], [2, 0], [2, 1]]


**Vì sao sai?** Đề nói điểm nguyên trong hình **gồm biên**; đoạn `[0, 2]` chứa **3** số nguyên chứ không phải 2. Thiếu `+1` làm trọng số sai và phân bố **không đều** (xem mục 12.6: thống kê bắt được). Đúng: `(x - a + 1) * (y - b + 1)`.

### Lỗi 2: chọn hình đều nhau (không theo số điểm)

Hình nhỏ bị "ưu ái": mỗi điểm của hình nhỏ có xác suất cao hơn điểm của hình lớn. Ta đã thấy ở mục 4 (`1/32` so với `1/30`) và mục 12.6.

### Lỗi 3: dùng `randrange` thay `randint` (lệch biên một đơn vị)

In [32]:
def wrong_randrange_pick(rects, rng):
    cnt = [(x - a + 1) * (y - b + 1) for a, b, x, y in rects]
    ends, run = [], 0
    for c in cnt:
        run += c
        ends.append(run)
    t = rng.randrange(1, ends[-1])                  # SAI: randrange không gồm ends[-1]
    i = bisect_left(ends, t)
    a, b, x, y = rects[i]
    return [rng.randrange(a, x), rng.randrange(b, y)]       # SAI: không bao giờ ra x, y (biên trên/phải)


rng = random.Random(5)
pts = {tuple(wrong_randrange_pick(rects_demo, rng)) for _ in range(5000)}
print("số điểm khác nhau:", len(pts), "(đúng phải là 31)")
print("có điểm biên trên-phải [1, 1] không?", (1, 1) in pts)

số điểm khác nhau: 17 (đúng phải là 31)
có điểm biên trên-phải [1, 1] không? False


**Vì sao sai?** `randrange(a, x)` cho `[a, x-1]` (không gồm `x`), nên không bao giờ ra biên trên/phải và tổng số điểm khả dĩ nhỏ hơn thực tế. Dùng `randint(a, x)` (gồm cả hai đầu), hoặc `randrange(a, x + 1)`.

### Lỗi 4: `bisect_left` / `bisect_right` nhầm với khoảng của `t`

Với `ends = [16, 31]` và `t ∈ [1, 31]` thì dùng **`bisect_left`**; với `t ∈ [0, 30]` thì dùng **`bisect_right`**. Dùng lẫn lộn làm sai một điểm ở ranh giới (t = 16), tức một điểm có xác suất sai.

In [33]:
ends = [16, 31]
print("t in [1,31] + bisect_left  : hình 0 có", sum(bisect_left(ends, t) == 0 for t in range(1, 32)), "giá trị t (đúng 16)")
print("t in [1,31] + bisect_right : hình 0 có", sum(bisect_right(ends, t) == 0 for t in range(1, 32)), "giá trị t (SAI, thiếu 1)")
print("t in [0,30] + bisect_right : hình 0 có", sum(bisect_right(ends, t) == 0 for t in range(0, 31)), "giá trị t (đúng 16)")

t in [1,31] + bisect_left  : hình 0 có 16 giá trị t (đúng 16)
t in [1,31] + bisect_right : hình 0 có 15 giá trị t (SAI, thiếu 1)
t in [0,30] + bisect_right : hình 0 có 16 giá trị t (đúng 16)


**Vì sao sai?** `bisect_right(ends, 16)` trả `1` (vì cần `ends[i] > 16`), đẩy `t = 16` sang hình 1: hình 0 chỉ còn 15 giá trị. Nhớ: `t` bắt đầu từ 1 ⇒ `bisect_left`; từ 0 ⇒ `bisect_right`.

### Lỗi 5: sinh danh sách điểm cho mọi hình (hết bộ nhớ)

Đã phân tích ở mục 5: `T` có thể `~4 × 10^8`. Đừng liệt kê điểm; chỉ lưu **số lượng**.

## 14. Mẹo & nhận dạng pattern

- **Nhận dạng:** "chọn ngẫu nhiên **đều** từ nhiều nhóm có kích thước khác nhau" ⇒ **chọn nhóm theo trọng số = kích thước**, rồi chọn đều trong nhóm. Cùng khung với: Random Pick with Weight (528), Random Pick Index (398), Linked List Random Node (382).
- **Prefix sum + bisect** là công thức chuẩn cho "lấy mẫu theo trọng số".
- **Đếm số nguyên trong `[lo, hi]` = `hi - lo + 1`.** Khi gặp "bao gồm biên", nhớ `+1`.
- **Song ánh số ↔ điểm** (`divmod`) biến "chọn điểm ngẫu nhiên" thành "chọn số ngẫu nhiên"; dùng được cho lưới, ma trận, hình hộp.
- **Test ngẫu nhiên:** cố định seed; kiểm tra **tính chất** (điểm hợp lệ) + **phân bố** (chi-bình-phương với ngưỡng rộng); thử cài đặt sai để biết test có đủ nhạy.
- Khi thấy "tối đa `10^4` lần gọi", hãy cân nhắc chi phí mỗi `pick`; `O(log k)` là quá đủ với `k <= 100`.

## 15. Biến thể

1. **Random Pick with Weight (528):** chỉ có trọng số, không có hình học — đúng Cách 3.
2. **Random Point in a Circle (478):** hình tròn; dùng lấy mẫu loại bỏ (rejection sampling) hoặc `r = R√U`.
3. **Pick không lặp lại** (mỗi điểm trả về tối đa một lần tới khi hết): dùng Fisher–Yates "ảo" bằng `dict` (bài tập 5).
4. **Điểm có thể thực (không nguyên):** trọng số là **diện tích** `(x-a)*(y-b)` (không `+1`), `u = uniform(a, x)`.
5. **Chỉ lấy điểm nội thất (không biên):** đếm `(x-a-1)*(y-b-1)` và dùng `randint(a+1, x-1)`.
6. **Hình chữ nhật có thể chồng lấn:** phải xử lý hợp (union) bằng quét đường (sweep line) hoặc loại bỏ trùng bằng rejection.

In [34]:
# Biến thể 5: điểm nội thất (không tính biên). Điều kiện: x-a >= 2 và y-b >= 2 (nếu không có thể không có điểm nội thất)
class PickInterior:
    def __init__(self, rects, rng=random):
        self.rng = rng
        self.rects = [r for r in rects if r[2] - r[0] >= 2 and r[3] - r[1] >= 2]
        self.ends, run = [], 0
        for a, b, x, y in self.rects:
            run += (x - a - 1) * (y - b - 1)
            self.ends.append(run)

    def pick(self):
        t = self.rng.randint(1, self.ends[-1])
        a, b, x, y = self.rects[bisect_left(self.ends, t)]
        return [self.rng.randint(a + 1, x - 1), self.rng.randint(b + 1, y - 1)]


pi = PickInterior([[0, 0, 4, 3], [10, 10, 11, 11], [20, 0, 24, 4]], random.Random(1))
print(pi.ends, [pi.pick() for _ in range(5)])

[6, 15] [[3, 1], [1, 2], [22, 2], [22, 1], [2, 1]]


## 16. So sánh các cách

| Cách | Ý tưởng | Time `pick` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê điểm | `choice` trên list mọi điểm | `O(1)` (init `O(T)`) | `O(T)` | chỉ khi `T` rất nhỏ; làm oracle |
| 2. Quét tuyến tính | trừ dần số điểm từng hình | `O(k)` | `O(k)` | `k` nhỏ, muốn đơn giản |
| 3. Cộng dồn + bisect | tìm nhị phân hình chứa `t` | `O(log k)` | `O(k)` | **nộp bài** |
| 4. Một số + `divmod` | `t` → hình → (hàng, cột) | `O(log k)` | `O(k)` | chỉ 1 lần gọi RNG; cần đánh số điểm (không lặp) |
| 5. Reservoir có trọng số | giữ hình `i` với xác suất `cnt_i / running` | `O(k)` | `O(k)` | dữ liệu dạng dòng, không biết trước `T` |

In [35]:
# Đối chiếu thực nghiệm: tỉ lệ rơi vào hình 0 của mọi cách xấp xỉ 16/31 = 0.516
random.seed(5)
N = 60000
for name, make in factories().items():
    obj = make(rects_demo, random.Random(77))
    in0 = sum(contains(rects_demo[0], obj.pick()) for _ in range(N))
    print(f"{name:<26}: {in0 / N:.4f} (kỳ vọng {16 / 31:.4f})")

Solution (global random)  : 0.5200 (kỳ vọng 0.5161)
Brute                     : 0.5139 (kỳ vọng 0.5161)


Scan                      : 0.5171 (kỳ vọng 0.5161)


Divmod                    : 0.5139 (kỳ vọng 0.5161)


Reservoir                 : 0.5173 (kỳ vọng 0.5161)


## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) số điểm nguyên của hình `[0, 0, 2, 1]`; (b) với `ends = [4, 13, 20]`, `t = 13` thì `bisect_left(ends, t)` bằng bao nhiêu? và `t = 14`?; (c) `divmod(7, 3)`; (d) `random.seed(5)` rồi hai lần `random.randint(1, 6)` có thể đổi giá trị nếu ta seed lại 5 trước lần thứ hai không?

**Bài 2 (tính tay).** Với `rects = [[0,0,1,1], [3,0,5,2]]`: tính `cnt`, `ends`, `T`; xác suất mỗi điểm; và dùng Cách 4 giải mã `t = 4` và `t = 12` (đánh số từ 0) thành điểm cụ thể.

**Bài 3 (điền TODO).** Hoàn thành lớp `PickTodo` theo Cách 3.

In [36]:
class PickTodo:
    def __init__(self, rects):
        # TODO 1: lưu rects; tạo self.ends là mảng cộng dồn của (x - a + 1) * (y - b + 1)
        self.ends = None

    def pick(self):
        # TODO 2: t = random.randint(1, self.ends[-1]); i = bisect_left(self.ends, t)
        # TODO 3: trả [randint(a, x), randint(b, y)] của hình i
        return None

# KHÔNG tạo đối tượng / gọi pick của lớp TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `PickDivmod2(rects, rng)` theo Cách 4 (một `randrange` + `divmod`) **không nhìn lại** mục 7; kiểm tra rằng `decode(t)` với `t` chạy hết `[0, T)` cho đúng `T` điểm phân biệt.

**Bài 5 (biến thể / mini challenge).** Viết `PickNoRepeat(rects, rng)` với `pick()` trả mỗi điểm nguyên **đúng một lần** (theo thứ tự ngẫu nhiên) cho tới khi hết `T` điểm (khi hết, `pick()` trả `None`). Yêu cầu `O(log k)` mỗi lần và bộ nhớ `O(số lần pick)` (không liệt kê `T` điểm). Gợi ý: **Fisher–Yates ảo** — một `dict` `swap` ánh xạ chỉ số đã bị hoán đổi.

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ).** Bài 1: nhớ `+1` cho biên; `bisect_left` trả vị trí đầu tiên có `ends[i] >= t`. Bài 3: sao chép đúng ba dòng của `Solution` ở mục 9.

**Mức 2 (vừa).** Bài 2: `cnt = [4, 9]`, `ends = [4, 13]`, mỗi điểm `1/13`. Với `t = 4`: `bisect_right([4,13], 4) = 1` (hình 1), `start = 13 - 9 = 4`, `offset = 0`. Bài 4: `decode(t)` dùng `bisect_right` vì `t` bắt đầu từ 0.

**Mức 3 (sát đáp án).** Bài 5: giữ `remaining = T`. Mỗi lần: `j = rng.randrange(remaining)`; kết quả là `swap.get(j, j)`; sau đó `remaining -= 1` và đặt `swap[j] = swap.get(remaining, remaining)` (đưa phần tử cuối vào chỗ trống). Điểm thứ `idx` được giải mã bằng `decode` ở Cách 4.

## 19. Đáp án tham khảo cho bài tập

In [37]:
# Bài 1
assert (2 - 0 + 1) * (1 - 0 + 1) == 6
ends1 = [4, 13, 20]
assert bisect_left(ends1, 13) == 1 and bisect_left(ends1, 14) == 2
assert divmod(7, 3) == (2, 1)
random.seed(5); first = [random.randint(1, 6) for _ in range(2)]
random.seed(5); again = [random.randint(1, 6) for _ in range(2)]
assert first == again                     # seed lại thì ra đúng dãy cũ

# Bài 2: rects = [[0,0,1,1],[3,0,5,2]] -> cnt [4, 9], ends [4, 13]
r2 = [[0, 0, 1, 1], [3, 0, 5, 2]]
d = PickDivmod(r2)
assert d.ends == [4, 13]
assert d.decode(4) == [3, 0]              # điểm đầu tiên của hình 1 (offset 0)
assert d.decode(12) == [5, 2]             # điểm cuối của hình 1 (offset 8 = hàng 2, cột 2)
print("bài 1-2 OK")

bài 1-2 OK


In [38]:
# Bài 3
class PickTodoAnswer:
    def __init__(self, rects):
        self.rects = rects
        self.ends = []
        total = 0
        for a, b, x, y in rects:
            total += (x - a + 1) * (y - b + 1)
            self.ends.append(total)

    def pick(self):
        t = random.randint(1, self.ends[-1])
        a, b, x, y = self.rects[bisect_left(self.ends, t)]
        return [random.randint(a, x), random.randint(b, y)]


random.seed(3)
o = PickTodoAnswer(rects_demo)
assert all(owners(rects_demo, o.pick()) == 1 for _ in range(500))
print("bài 3 OK")

bài 3 OK


In [39]:
# Bài 4
class PickDivmod2:
    def __init__(self, rects, rng=random):
        self.rng, self.rects = rng, rects
        self.ends, run = [], 0
        for a, b, x, y in rects:
            run += (x - a + 1) * (y - b + 1)
            self.ends.append(run)

    def decode(self, t):
        i = bisect_right(self.ends, t)
        a, b, x, y = self.rects[i]
        w = x - a + 1
        start = self.ends[i] - w * (y - b + 1)
        row, col = divmod(t - start, w)
        return [a + col, b + row]

    def pick(self):
        return self.decode(self.rng.randrange(self.ends[-1]))


for rr in (rects_demo, rects3, r2):
    q = PickDivmod2(rr)
    pts = {tuple(q.decode(t)) for t in range(q.ends[-1])}
    assert len(pts) == q.ends[-1] == count_points(rr)
    assert all(owners(rr, p) == 1 for p in pts)
print("bài 4 OK")

bài 4 OK


In [40]:
# Bài 5: Fisher-Yates ảo
class PickNoRepeat:
    def __init__(self, rects, rng=random):
        self.dec = PickDivmod2(rects, rng)
        self.rng = rng
        self.remaining = self.dec.ends[-1]
        self.swap = {}

    def pick(self):
        if self.remaining == 0:
            return None
        j = self.rng.randrange(self.remaining)
        idx = self.swap.get(j, j)
        self.remaining -= 1
        self.swap[j] = self.swap.get(self.remaining, self.remaining)
        return self.dec.decode(idx)


for rr, seed in ((rects_demo, 1), (rects3, 2), (r2, 3)):
    nr = PickNoRepeat(rr, random.Random(seed))
    T = count_points(rr)
    out = [nr.pick() for _ in range(T)]
    assert len({tuple(p) for p in out}) == T and nr.pick() is None
    assert all(owners(rr, p) == 1 for p in out)
print("bài 5 OK")

bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (chọn theo trọng số bằng cộng dồn + bisect)

```python
import random
from bisect import bisect_left

class Solution:
    def __init__(self, rects: List[List[int]]):
        self.rects = rects
        self.ends = []
        total = 0
        for a, b, x, y in rects:
            total += (x - a + 1) * (y - b + 1)
            self.ends.append(total)

    def pick(self) -> List[int]:
        t = random.randint(1, self.ends[-1])
        a, b, x, y = self.rects[bisect_left(self.ends, t)]
        return [random.randint(a, x), random.randint(b, y)]
```

**Checklist**

- [ ] Điểm nguyên **gồm biên** ⇒ số điểm `(x - a + 1) * (y - b + 1)`?
- [ ] Xác suất chọn hình **tỉ lệ số điểm** (không phải 1/k)?
- [ ] `randint` (gồm hai đầu) cho tọa độ; `t ∈ [1, T]` đi với `bisect_left`?
- [ ] Không liệt kê điểm (tránh MLE)?
- [ ] Đã test: điểm nằm trong đúng một hình, cùng seed ⇒ cùng dãy, chi-bình-phương trên rects nhỏ, hình ở `±10^9`, hình `2x2`?
- [ ] Đã thử một cài đặt sai (quên `+1`, chọn hình đều) để chắc test đủ nhạy?